# 小体系定义

该文档将确定一些典型的体系与计算方法，为后续的开发作准备。

该文档一般来说不适合多次执行。

In [1]:
from pyscf import gto, scf, dft, lib
import numpy as np
lib.num_threads(16)

16

In [2]:
def save_array(name, mf, mf_hess):
    arrays = {
        "mo_coeff": np.asarray(mf.mo_coeff, order="C"),
        "mo_occ": np.asarray(mf.mo_occ, order="C"),
        "mo_energy": np.asarray(mf.mo_energy, order="C"),
        "rdm1": mf.make_rdm1(),
        "ref_de": mf_hess.de
    }
    if hasattr(mf, "grids"):
        grids = mf.grids
        arrays["grid_coords"] = grids.coords
        arrays["grid_weights"] = grids.weights
    np.savez(name, **arrays)

## 小体系：NH3 (restricted)

In [3]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [4]:
%%time
mf_hf = scf.RHF(mol).density_fit().run()

converged SCF energy = -56.1132387662624
CPU times: user 501 ms, sys: 14.1 ms, total: 515 ms
Wall time: 36.8 ms


In [5]:
%%time
mf_hf_hess = mf_hf.Hessian().run()

CPU times: user 3.51 s, sys: 178 ms, total: 3.69 s
Wall time: 240 ms


In [6]:
%%time
mf_b3lyp = dft.RKS(mol, xc="B3LYP").density_fit().run()

converged SCF energy = -56.4997024923723
CPU times: user 2.63 s, sys: 516 ms, total: 3.15 s
Wall time: 216 ms


In [7]:
%%time
mf_b3lyp_hess = mf_b3lyp.Hessian().run()

CPU times: user 17.1 s, sys: 1.82 s, total: 18.9 s
Wall time: 1.22 s


In [8]:
%%time
mf_tpss0 = dft.RKS(mol, xc="TPSS0").density_fit().run()

converged SCF energy = -56.4847025937838
CPU times: user 3.99 s, sys: 664 ms, total: 4.65 s
Wall time: 301 ms


In [9]:
%%time
mf_tpss0_hess = mf_tpss0.Hessian().run()


WARN: MGGA Hessian is sensitive to dft grids. grids.level 3 may not be dense enough.

CPU times: user 35 s, sys: 4.81 s, total: 39.8 s
Wall time: 2.56 s


In [10]:
save_array("nh3_r_hf", mf_hf, mf_hf_hess)
save_array("nh3_r_b3lyp", mf_b3lyp, mf_b3lyp_hess)
save_array("nh3_r_tpss0", mf_tpss0, mf_tpss0_hess)

## 中等体系：CH2CH3OH (restricted)

In [11]:
xyz = """
C          0.91993       -0.00984        0.04649
C          2.43421       -0.01198        0.05340
O          2.91590       -1.03737        0.90936
H          0.53513        0.76934       -0.61730
H          0.52793        0.16086        1.05450
H          0.53445       -0.97954       -0.28537
H          2.82564        0.94892        0.40128
H          2.82120       -0.19808       -0.95268
H          2.57632       -0.85751        1.80258
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [12]:
%%time
mf_hf = scf.RHF(mol).density_fit().run()

converged SCF energy = -154.144103877433
CPU times: user 1.83 s, sys: 105 ms, total: 1.94 s
Wall time: 127 ms


In [13]:
%%time
mf_hf_hess = mf_hf.Hessian().run()

CPU times: user 1min 3s, sys: 3.21 s, total: 1min 6s
Wall time: 4.29 s


In [14]:
%%time
mf_b3lyp = dft.RKS(mol, xc="B3LYP").density_fit().run()

converged SCF energy = -155.108575764807
CPU times: user 15.1 s, sys: 2.68 s, total: 17.8 s
Wall time: 1.15 s


In [15]:
%%time
mf_b3lyp_hess = mf_b3lyp.Hessian().run()

CPU times: user 4min 57s, sys: 32.7 s, total: 5min 30s
Wall time: 21.9 s


In [16]:
%%time
mf_tpss0 = dft.RKS(mol, xc="TPSS0").density_fit().run()

converged SCF energy = -155.086038988593
CPU times: user 21.9 s, sys: 3.42 s, total: 25.3 s
Wall time: 1.75 s


In [17]:
%%time
mf_tpss0_hess = mf_tpss0.Hessian().run()


WARN: MGGA Hessian is sensitive to dft grids. grids.level 3 may not be dense enough.

CPU times: user 9min 40s, sys: 1min 26s, total: 11min 6s
Wall time: 43.9 s


In [18]:
save_array("et_r_hf", mf_hf, mf_hf_hess)
save_array("et_r_b3lyp", mf_b3lyp, mf_b3lyp_hess)
save_array("et_r_tpss0", mf_tpss0, mf_tpss0_hess)

## 小体系：NH3 (unrestricted)

In [19]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", charge=2, spin=2, max_memory=32000).build()

In [20]:
%%time
mf_hf = scf.UHF(mol).density_fit().run()

converged SCF energy = -54.839029292921  <S^2> = 2.0132493  2S+1 = 3.0088199
CPU times: user 1.13 s, sys: 26.3 ms, total: 1.15 s
Wall time: 74.2 ms


In [21]:
%%time
mf_hf_hess = mf_hf.Hessian().run()

CPU times: user 4.31 s, sys: 113 ms, total: 4.42 s
Wall time: 286 ms


In [22]:
%%time
mf_b3lyp = dft.UKS(mol, xc="B3LYP").density_fit().run()

converged SCF energy = -55.1081864477309  <S^2> = 2.003217  2S+1 = 3.0021439
CPU times: user 7.11 s, sys: 1.01 s, total: 8.12 s
Wall time: 527 ms


In [23]:
%%time
mf_b3lyp_hess = mf_b3lyp.Hessian().run()

CPU times: user 35.4 s, sys: 2.48 s, total: 37.9 s
Wall time: 2.47 s


In [24]:
%%time
mf_tpss0 = dft.UKS(mol, xc="TPSS0").density_fit().run()

converged SCF energy = -55.1103711749515  <S^2> = 2.0045647  2S+1 = 3.0030416
CPU times: user 11.8 s, sys: 1.1 s, total: 12.9 s
Wall time: 835 ms


In [25]:
%%time
mf_tpss0_hess = mf_tpss0.Hessian().run()


WARN: MGGA Hessian is sensitive to dft grids. grids.level 3 may not be dense enough.

CPU times: user 1min 5s, sys: 2.33 s, total: 1min 7s
Wall time: 4.33 s


In [26]:
save_array("nh3_u_hf", mf_hf, mf_hf_hess)
save_array("nh3_u_b3lyp", mf_b3lyp, mf_b3lyp_hess)
save_array("nh3_u_tpss0", mf_tpss0, mf_tpss0_hess)

## 中等体系：CH3CH2OH (unrestricted)

In [27]:
xyz = """
C          0.91993       -0.00984        0.04649
C          2.43421       -0.01198        0.05340
O          2.91590       -1.03737        0.90936
H          0.53513        0.76934       -0.61730
H          0.52793        0.16086        1.05450
H          0.53445       -0.97954       -0.28537
H          2.82564        0.94892        0.40128
H          2.82120       -0.19808       -0.95268
H          2.57632       -0.85751        1.80258
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", charge=2, spin=2, max_memory=32000).build()

In [28]:
%%time
mf_hf = scf.UHF(mol).density_fit().run()

converged SCF energy = -153.119496284381  <S^2> = 2.0274131  2S+1 = 3.0182201
CPU times: user 7.83 s, sys: 419 ms, total: 8.25 s
Wall time: 778 ms


In [29]:
%%time
mf_hf_hess = mf_hf.Hessian().run()

CPU times: user 1min 49s, sys: 3.94 s, total: 1min 53s
Wall time: 7.29 s


In [30]:
%%time
mf_b3lyp = dft.UKS(mol, xc="B3LYP").density_fit().run()

converged SCF energy = -154.024687124138  <S^2> = 2.0072334  2S+1 = 3.0048184
CPU times: user 36.8 s, sys: 4.24 s, total: 41 s
Wall time: 2.63 s


In [31]:
%%time
mf_b3lyp_hess = mf_b3lyp.Hessian().run()

CPU times: user 10min 3s, sys: 1min 8s, total: 11min 11s
Wall time: 43.6 s


In [32]:
%%time
mf_tpss0 = dft.UKS(mol, xc="TPSS0").density_fit().run()

converged SCF energy = -154.007467940646  <S^2> = 2.0106931  2S+1 = 3.0071203
CPU times: user 1min 6s, sys: 7.68 s, total: 1min 14s
Wall time: 4.77 s


In [33]:
%%time
mf_tpss0_hess = mf_tpss0.Hessian().run()


WARN: MGGA Hessian is sensitive to dft grids. grids.level 3 may not be dense enough.

CPU times: user 21min 32s, sys: 3min 30s, total: 25min 3s
Wall time: 1min 37s


In [34]:
save_array("et_u_hf", mf_hf, mf_hf_hess)
save_array("et_u_b3lyp", mf_b3lyp, mf_b3lyp_hess)
save_array("et_u_tpss0", mf_tpss0, mf_tpss0_hess)